**Hücre 1:** Hugging Face Kimlik Doğrulama (Authentication)

**Açıklama:** Bu hücre, projemizde kullanacağımız **Llama-3** gibi kapalı (gated) modellere erişim sağlamak amacıyla Hugging Face platformu üzerinden kimlik doğrulaması yapar. Modelin ağırlıklarını güvenli bir şekilde indirmek için önceden oluşturulan erişim anahtarı (token) kullanılır.

In [ ]:
from huggingface_hub import login, whoami

# Hugging Face platformuna erişim sağlamak için kimlik doğrulama işlemi.
# Llama-3 gibi kısıtlı modellere erişim izni almak için bu adım gereklidir.

# Lütfen 'hf_...' ile başlayan kendi erişim anahtarınızı (token) aşağıya yapıştırın.
my_token = "YOUR_HF_TOKEN_HERE"

try:
    # 1. Giriş yapmayı deniyoruz
    login(token=my_token)

    # 2. Token'ın geçerliliğini doğrulamak için kullanıcı bilgilerini sorguluyoruz
    user_info = whoami(token=my_token)

    # 3. Başarılı olursa kullanıcı adını ve onay mesajını yazdırıyoruz
    print(f"\n✅ Giriş Başarılı!. (Login Successful)")

except Exception as e:
    # 4. Herhangi bir hata olursa (token yanlışsa vb.) hata mesajı yazdırıyoruz
    print(f"\n❌ Giriş Başarısız! Lütfen tokeninizi kontrol edin.")
    print(f"Hata Detayı: {e}")


✅ Giriş Başarılı!. (Login Successful)


**Hücre 2:** Veri Seti Hazırlığı ve Otel Politikası Tanımlama

**Açıklama:** Bu hücre, projemizin temel bilgi kaynağını oluşturur. İlk bölümde, otel hizmetlerini ve kurallarını içeren ham metin verisi **(Sunrise Hotel Politika Dokümanı)** tanımlanmıştır. İkinci bölümde ise, modelin otel hakkındaki soruları belirli bir üslupla yanıtlamasını sağlamak amacıyla bu metinden türetilen 20 adet soru-cevap çiftinden oluşan Fine-Tuning **(İnce Ayar)** veri seti hazırlanmıştır. Hazırlanan bu veriler, Hugging Face datasets formatına dönüştürülerek eğitim sürecine hazır hale getirilmiştir.

In [2]:
# 1. OTEL POLİTİKA VE HİZMET DOKÜMANI TANIMLAMA
# Bu metin, hem RAG sisteminde kaynak olarak kullanılacak hem de eğitim verilerinin temelini oluşturacaktır.
hotel_policy_text = """
OTEL POLİTİKA VE HİZMET DOKÜMANI
Bu doküman, Sunrise Hotel’in (örnek otel) müşterilerine sunduğu hizmetleri, kuralları, rezervasyon, iptal, iade, giriş-çıkış işlemleri ve diğer tüm önemli bilgileri içmektedir.
1. GENEL BİLGİLER
Sunrise Hotel, şehir merkezine yakın konumda bulunan ve hem iş hem de tatil amaçlı konaklamalar için uygun bir tesistir. Otelimiz 120 oda, restoran, spa merkezi, toplantı salonları ve açık yüzme havuzuna sahiptir.
Otel, yılın 365 günü hizmet vermektedir.
2. REZERVASYON POLİTİKASI
• Rezervasyonlar otelin resmi web sitesi, telefon veya e-posta yoluyla yapılabilir.
• Rezervasyon sırasında misafirlerden ad, soyad, telefon numarası ve ödeme bilgileri talep edilir.
• Tüm rezervasyonlar, otel tarafından e-posta veya SMS ile onaylanır.
• Rezervasyonun kesinleşmesi için en az %20 ön ödeme alınmaktadır.
• Erken rezervasyonlarda %10 indirim uygulanır.
3. GİRİŞ VE ÇIKIŞ SAATLERİ
• Otele giriş (check-in) saati: 14:00
• Otelden çıkış (check-out) saati: 11:00
• Erken giriş veya geç çıkış talepleri müsaitlik durumuna göre kabul edilir.
• Geç çıkış yapan misafirlerden ek ücret talep edilebilir.
4. İPTAL VE İADE POLİTİKASI
• Rezervasyon, giriş tarihinden en az 48 saat önce iptal edilirse ön ödeme iade edilir.
• Son 48 saat içinde yapılan iptallerde ön ödeme iade edilmez.
• Otele gelinmemesi (no-show) durumunda ilk gece ücreti tahsil edilir.
• Mücbir sebepler (doğal afet, sağlık sorunları vb.) durumunda, iade otel yönetiminin onayına bağlıdır.
5. ODA ÖZELLİKLERİ VE HİZMETLER
Tüm odalarda: Ücretsiz Wi-Fi, Klimа, Televizyon, Mini bar, Güvenlik kasası, Özel banyo ve havlu seti bulunmaktadır.
Suit odalarda ek olarak: Jakuzi, Şehir manzaralı balkon, Ücretsiz meyve sepeti.
6. TEMİZLİK VE ODA SERVİSİ
• Odalar her gün saat 09:00 - 16:00 arasında temizlenir.
• Havlu ve çarşaflar iki günde bir değiştirilir.
• Oda servisi 07:00 - 23:00 saatleri arasında hizmet verir.
7. RESTORAN VE YEMEK SAATLERİ
• Kahvaltı: 07:00 – 10:00, Öğle yemeği: 12:00 – 15:00, Akşam yemeği: 18:00 – 22:00.
• Vejetaryen ve vegan menü seçenekleri mevcuttur.
8. SPA VE HAVUZ KULLANIMI
• Spa merkezi: 10:00 – 20:00, Havuz: 08:00 – 19:00.
• 12 yaşından küçük çocukların spa alanına girmesi yasaktır. Bone kullanımı tavsiye edilir.
9. EVCİL HAYVAN POLİTİKASI
• Küçük ırk evcil hayvanlara izin verilmektedir. Günlük ek 20€ temizlik ücreti uygulanır.
10. ÖDEME YÖNTEMLERİ
• Kredi kartı (Visa, MasterCard), Nakit, Havale / EFT. American Express kabul edilmemektedir.
11. MÜŞTERİ SORUMLULUKLARI
• Zarar gören eşyaların ücreti müşteriden tahsil edilir. Gürültü yapmak yasaktır.
12. İLETİŞİM BİLGİLERİ
Telefon: +90 555 555 55 55, E-posta: smart2025hotel@gmail.com
"""

# 2. MODELİ EĞİTMEK İÇİN KULLANILACAK GENİŞLETİLMİŞ VERİ SETİ (FINE-TUNING DATASET)
# Modelin otel personeli gibi doğru ve tutarlı cevaplar vermesi için hazırlanan soru-cevap çiftleri.
train_data = [
    # --- Genel Bilgiler ---
    {"instruction": "Sunrise Hotel'in konumu ve genel imkanları nelerdir?", "input": "", "output": "Sunrise Hotel şehir merkezine yakın konumdadır. 120 oda, restoran, spa merkezi, toplantı salonları ve açık havuz imkanları sunar."},
    {"instruction": "Otel yılın hangi zamanlarında hizmet veriyor?", "input": "", "output": "Otelimiz yılın 365 günü kesintisiz hizmet vermektedir."},

    # --- Rezervasyon ---
    {"instruction": "Nasıl rezervasyon yapabilirim?", "input": "", "output": "Resmi web sitemiz, telefon numaramız veya e-posta adresimiz üzerinden rezervasyon yapabilirsiniz."},
    {"instruction": "Rezervasyonun kesinleşmesi için ön ödeme gerekiyor mu?", "input": "", "output": "Evet, rezervasyonun kesinleşmesi için en az %20 oranında ön ödeme alınmaktadır."},
    {"instruction": "Erken rezervasyon yapanlar için bir avantaj var mı?", "input": "", "output": "Evet, erken rezervasyonlarda %10 indirim uygulanmaktadır."},

    # --- Giriş ve Çıkış ---
    {"instruction": "Sunrise Hotel'e giriş ve çıkış saatleri nedir?", "input": "", "output": "Otele giriş (check-in) saati 14:00, çıkış (check-out) saati ise 11:00'dir."},
    {"instruction": "Geç çıkış yaparsam ücret öder miyim?", "output": "Müsaitlik durumuna göre geç çıkış talepleri kabul edilebilir ancak ek ücret talep edilebilir."},

    # --- İptal ve İade ---
    {"instruction": "Rezervasyonumu iptal edersem paramı geri alabilir miyim?", "input": "", "output": "Giriş tarihinden en az 48 saat önce iptal ederseniz ön ödeme iade edilir. Son 48 saatteki iptallerde iade yapılmaz."},
    {"instruction": "Otele gelemezsem (no-show) ne kadar ücret kesilir?", "input": "", "output": "Otele gelinmemesi (no-show) durumunda ilk gece ücreti tahsil edilmektedir."},

    # --- Oda Özellikleri ---
    {"instruction": "Standart odalarda hangi imkanlar mevcut?", "input": "", "output": "Tüm odalarda ücretsiz Wi-Fi, klima, TV, mini bar, güvenlik kasası ve özel banyo bulunmaktadır."},
    {"instruction": "Suit odaların standart odalardan farkı nedir?", "input": "", "output": "Suit odalarda ek olarak jakuzi, şehir manzaralı balkon ve ücretsiz meyve sepeti hizmeti sunulmaktadır."},

    # --- Temizlik ve Servis ---
    {"instruction": "Oda temizliği hangi saatlerde yapılıyor?", "input": "", "output": "Odalarımız her gün saat 09:00 ile 16:00 arasında titizlikle temizlenmektedir."},
    {"instruction": "Oda servisi 24 saat açık mı?", "input": "", "output": "Hayır, oda servisi 07:00 - 23:00 saatleri arasında hizmet vermektedir."},

    # --- Restoran ---
    {"instruction": "Kahvaltı, öğle ve akşam yemeği saatleri nedir?", "input": "", "output": "Kahvaltı 07:00-10:00, öğle yemeği 12:00-15:00, akşam yemeği ise 18:00-22:00 saatleri arasındadır."},
    {"instruction": "Restoranda vegan seçenekler var mı?", "input": "", "output": "Evet, otel restoranımızda vejetaryen ve vegan menü seçenekleri mevcuttur."},

    # --- Spa ve Havuz ---
    {"instruction": "Spa ve havuz kullanım saatleri nedir?", "input": "", "output": "Spa merkezi 10:00-20:00, havuz ise 08:00-19:00 saatleri arasında hizmet vermektedir."},
    {"instruction": "Çocuklar spa alanını kullanabilir mi?", "input": "", "output": "12 yaşından küçük çocukların spa alanına girmesi yasaktır."},

    # --- Evcil Hayvan ---
    {"instruction": "Otelde evcil hayvan kabul ediliyor mu?", "input": "", "output": "Küçük ırk evcil hayvanlara izin verilmektedir. Günlük ek 20€ temizlik ücreti uygulanır ve restoran/havuz alanına girmeleri yasaktır."},

    # --- Ödeme ---
    {"instruction": "Hangi ödeme yöntemlerini kabul ediyorsunuz?", "input": "", "output": "Kredi kartı (Visa, MasterCard), nakit ve Havale/EFT kabul edilir. American Express kabul edilmemektedir."},

    # --- İletişيم ---
    {"instruction": "Otelin iletişim bilgileri nelerdir?", "input": "", "output": "Telefon: +90 555 555 55 55, E-posta: smart2025hotel@gmail.com. Adresimiz İstanbul, Örnek Mah. Tatil Cad. No:25'tir."}
]

# 3. VERİLERİ EĞİTİM FORMATINA DÖNÜŞTÜRME
# Python listesini, eğitim kütüphanelerinin anlayacağı 'Dataset' formatına çeviriyoruz.
from datasets import Dataset
dataset = Dataset.from_list(train_data)

print("Veri işleme başarıyla tamamlandı! Örnek sayısı:", len(dataset))

Veri işleme başarıyla tamamlandı! Örnek sayısı: 20


**Hücre 3: Model Yükleme ve LoRA Yapılandırması**

**Açıklama:** Bu hücre, projenin teknik temelini oluşturur. İlk olarak, büyük dil modellerini (LLM) verimli bir şekilde eğitmek için gerekli olan **Unsloth**, **Xformers** ve **PEFT** gibi kütüphanelerin kurulumu gerçekleştirilir. Ardından, **Llama-3-8B-Instruct** modeli, bellek kullanımını minimize etmek için **4-bit kuantizasyon** yöntemiyle yüklenir. Son aşamada, modeli tüm parametreleri yerine sadece küçük bir kısmını eğiterek optimize eden **LoRA (Low-Rank Adaptation)** mimarisi yapılandırılır. Bu yöntem, sınırlı donanım kaynaklarıyla yüksek performanslı "İnce Ayar" (Fine-Tuning) yapmamıza olanak tanır.

**Hücre 3.1: Kütüphane Kurulumu (Çıktılar Gizli)**

In [ ]:
# 1. KÜTÜPHANE KURULUMU
# Kurulum çıktıları çok uzun olduğu için '%%capture' ile gizlenmiştir.
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

**Hücre 3.2: Model Yükleme ve LoRA Ayarları (Çıktılar Görünür)**

In [4]:
# 2. GEREKLİ MODÜLLERİN İÇE AKTARILMASI
from unsloth import FastLanguageModel
import torch

# 3. ANA MODEL VE TOKENIZER'IN YÜKLENMESİ
# Bellek verimliliği için 4-bit yükleme (load_in_4bit) aktif edilmiştir.
max_seq_length = 2048
dtype = None
load_in_4bit = True

print("⏳ Jupyter Notebook: Model ve tokenizer yükleniyor... (Loading model...)")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# 4. DURDURMA TOKEN'ININ TANIMLANMASI
EOS_TOKEN = tokenizer.eos_token

# 5. LORA (LOW-RANK ADAPTATION) YAPILANDIRMASI
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

print("✅ Her şey hazır! Kütüphane kuruldu ve Model yüklendi. (Ready!)")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
⏳ Jupyter Notebook: Model ve tokenizer yükleniyor... (Loading model...)
==((====))==  Unsloth 2025.12.9: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/345 [00:00<?, ?B/s]

Unsloth 2025.12.9 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


✅ Her şey hazır! Kütüphane kuruldu ve Model yüklendi. (Ready!)


**Hücre 4: Model Eğitimi (Fine-Tuning) ve Kayıt İşlemi**

**Açıklama:** Bu hücre, modelin Sunrise Hotel bilgilerini "öğrendiği" asıl eğitim aşamasını gerçekleştirir. İlk olarak, ham veriler **Alpaca** formatına (Talimat-Giriş-Yanıt yapısı) dönüştürülür; böylece model, kullanıcı soruları ile doküman verileri arasındaki ilişkiyi kavrar. Eğitim sürecinde SFTTrainer (Supervised Fine-Tuning) kullanılarak **3 tam tur (epoch)** boyunca veriler üzerinden geçilir. Eğitim tamamlandıktan sonra, elde edilen düşük dereceli adaptörler **(LoRA adapters)** yerel bir klasöre kaydedilir. Bu işlem, modelin genel yeteneklerini bozmadan sadece özel bilgileri öğrenmesini sağlayarak sistemin bir "karar destek aracı" olarak uzmanlaşmasını sağlar.

In [5]:
import os
import psutil
import builtins


builtins.psutil = psutil
# ---------------------------

# 1. WANDB TAKİBİNİ DEVRE DIŞI BIRAKMA
os.environ["WANDB_DISABLED"] = "true"

# 2. PROMPT ŞABLONU VE TOKEN TANIMLAMA
alpaca_prompt = """Aşağıda bir görevi tanımlayan bir talimat yer almaktadır. İsteği uygun şekilde tamamlayan bir yanıt yazınız.

### Talimat:
{}

### Giriş:
{}

### Yanıt:
{}"""

EOS_TOKEN = tokenizer.eos_token

# 3. VERİ FORMATLAMA FONKSİYONU
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True,)

# 4. EĞİTİMCİ (TRAINER) YAPILANDIRMASI
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

# 5. EĞİTİMİN BAŞLATILMASI
print("Eğitim süreci başlıyor... (Training is starting...)")
trainer_stats = trainer.train()
print("Eğitim başarıyla tamamlandı! (Training finished!)")

# 6. EĞİTİLEN LORA AĞIRLIKLARININ KAYDEDİLMESİ
model.save_pretrained("sunrise_hotel_lora_model")
tokenizer.save_pretrained("sunrise_hotel_lora_model")
print("Model 'sunrise_hotel_lora_model' klasörüne kaydedildi.")

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/20 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.


Eğitim süreci başlıyor... (Training is starting...)


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 20 | Num Epochs = 3 | Total steps = 9
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss
1,3.224600
2,2.878800
3,3.055100
4,2.772600
5,2.145400
6,1.929300
7,1.493700
8,1.190500
9,0.839400


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Eğitim başarıyla tamamlandı! (Training finished!)
Model 'sunrise_hotel_lora_model' klasörüne kaydedildi.


**Hücre 5: RAG (Retrieval-Augmented Generation) Sisteminin Kurulumu**

**Açıklama:** Bu hücre, projenin ikinci ana bileşeni olan **RAG (Arama Destekli Üretim)** sistemini yapılandırır. RAG, modelin sadece eğitim sırasında öğrendiği bilgilere güvenmek yerine, yanıt üretmeden önce orijinal doküman içerisinde gerçek zamanlı arama yapmasına olanak tanır. Süreç şu şekilde işler: Otel politikası küçük parçalara **(chunks)** ayrılır, bu parçalar çok dilli bir yapay zeka modeli kullanılarak sayısal vektörlere **(embeddings)** dönüştürülür ve **FAISS** kütüphanesi ile hızlı bir arama indeksi oluşturulur. Bu sayede sistem, kullanıcı sorusuna en yakın olan orijinal metin parçalarını saniyeler içinde bularak modele "kanıt" olarak sunar.

In [6]:
# 1. GEREKLİ RAG KÜTÜPHANELERİNİN KURULUMU
# Metinleri vektöre çevirmek için sentence-transformers, hızlı arama için FAISS kullanıyoruz.
!pip install -U sentence-transformers faiss-cpu

from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# 2. DOKÜMANIN PARÇALARA AYRILMASI (CHUNKING)
# Politika metni, daha hassas arama yapılabilmesi için satır bazlı küçük parçalara bölünür.
# 10 karakterden kısa olan boş veya anlamsız satırlar filtrelenir.
policy_chunks = [chunk.strip() for chunk in hotel_policy_text.split('\n') if len(chunk.strip()) > 10]

# 3. VEKTÖRLEŞTİRME MODELİNİN YÜKLENMESİ (EMBEDDING MODEL)
# Türkçe dil desteği olan, hafif ve hızlı 'paraphrase-multilingual-MiniLM-L12-v2' modeli seçilmiştir.
embed_model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

# 4. METİN PARÇALARININ SAYISAL VEKTÖRLERE DÖNÜŞTÜRÜLMESİ
# Her bir metin parçası, anlamını temsil eden matematiksel bir vektöre çevrilir.
chunk_embeddings = embed_model.encode(policy_chunks)

# 5. FAISS VEKTÖR İNDEKSİNİN OLUŞTURULMASI
# Vektörlerin boyutuna göre bir FAISS indeksi oluşturulur ve veriler bu indekse eklenir.
dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(chunk_embeddings).astype('float32'))

# 6. İLGİLİ BAĞLAM GETİRME FONKSİYONU (RETRIEVAL)
def get_relevant_context(query, k=3):
    """Kullanıcının sorusuna en yakın olan en alakalı 3 metin parçasını dokümandan bulur."""
    # Soruyu vektöre çeviriyoruz
    query_embedding = embed_model.encode([query])
    # İndeks üzerinde benzerlik araması yapıyoruz
    distances, indices = index.search(np.array(query_embedding).astype('float32'), k)

    # Bulunan en alakalı parçaları birleştirerek bir bağlam (context) oluşturuyoruz
    context = ""
    for i in indices[0]:
        if i != -1: # Geçerli bir sonuç olup olmadığını kontrol ediyoruz
            context += policy_chunks[i] + "\n"
    return context

print("✅ RAG Sistemi Hazır! (RAG System is Ready!)")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 23.4 MB/s eta 0:00:00


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ RAG Sistemi Hazır! (RAG System is Ready!)


**Hücre 6: Temel Model Çıkarım (Inference) Fonksiyonu**

**Açıklama:** Bu hücre, ince ayar yapılmış (fine-tuned) modelin performansını doğrudan test etmek için kullanılan çıkarım fonksiyonunu tanımlar. **FastLanguageModel.for_inference** komutu, modeli çıkarım moduna alarak hızlandırır ve bellek kullanımını optimize eder. Fonksiyon, kullanıcıdan gelen soruyu eğitim aşamasında kullanılan **Alpaca** şablonuna yerleştirir, belirteçlere (tokens) dönüştürür ve modelin eğitilmiş ağırlıklarını kullanarak bir yanıt üretir. Bu aşama, RAG desteği olmadan modelin kendi "hafızasındaki" bilgileri ne kadar doğru yansıttığını gözlemlemek için gereklidir.

In [7]:
# 1. ÇIKARIM MODUNUN AKTİF EDİLMESİ
# Modeli çıkarım (inference) moduna alarak üretim hızını 2 kat artırıyoruz.
FastLanguageModel.for_inference(model)

def ask_model(question):
    """
    Sadece ince ayar yapılmış (fine-tuned) modeli kullanarak yanıt üretir.
    Bu aşamada harici bir doküman araması (RAG) yapılmaz.
    """
    # 2. PROMPT ŞABLONUNUN HAZIRLANMASI
    # Modelin eğitimde öğrendiği yapıyı (Talimat-Giriş-Yanıt) burada da uyguluyoruz.
    prompt = alpaca_prompt.format(
        question, # Kullanıcının sorusu
        "",       # Giriş alanı (Fine-tuning testi için boş bırakılır)
        "",       # Yanıt alanı (Model tarafından doldurulacaktır)
    )

    # 3. METNİN SAYISAL VERİYE (TOKEN) DÖNÜŞTÜRÜLMESİ
    # Metin, GPU üzerinde işlenebilmesi için tensör formatına çevrilir.
    inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

    # 4. YANITIN ÜRETİLMESİ (GENERATION)
    # Model, kendisine verilen girdiyi temel alarak en uygun yanıtı oluşturur.
    outputs = model.generate(
        **inputs,
        max_new_tokens = 128, # Üretilecek yanıtın maksimum uzunluğu
        use_cache = True,     # İşlemi hızlandırmak için önbellek kullanılır
    )

    # 5. YANITIN ÇÖZÜMLENMESİ VE TEMİZLENMESİ
    # Sayısal çıktılar tekrar okunabilir metne dönüştürülür.
    decoded_output = tokenizer.batch_decode(outputs)[0]

    # Metindeki şablon kısımları temizlenerek sadece modelin verdiği gerçek yanıt döndürülür.
    response = decoded_output.split("### Yanıt:\n")[-1].replace(EOS_TOKEN, "").strip()
    return response

print("Model teste hazır! (Model is ready for testing)")

Model teste hazır! (Model is ready for testing)


**Hücre 7: İnce Ayar Yapılmış Modelin Manuel Testi**

**Açıklama:** Bu hücre, eğitim sürecinden (Fine-tuning) sonra modelin Sunrise Hotel hakkındaki bilgileri ne kadar iyi öğrendiğini test etmek amacıyla kullanılır. Burada model, harici bir dokümandan yardım almadan (RAG sistemi devre dışıyken), sadece kendi içsel parametrelerinde kayıtlı olan bilgileri kullanarak bir cevap üretir. Bu test, eğitim başarısını doğrulamak ve modelin "hafızasındaki" bilgilerin doğruluğunu kontrol etmek için kritik bir adımdır.

In [8]:
# 1. TEST SORUSUNUN BELİRLENMESİ
# Modelin eğitim verisinde yer alan bir bilgiyi hafızasından çağırıp çağıramadığını kontrol ediyoruz.
test_question = "Sunrise Hotel'e giriş ve çıkış saatleri nedir?"

# 2. MODELİN YANITLAMASI
# Önceki hücrede tanımlanan 'ask_model' fonksiyonunu kullanarak yanıtı alıyoruz.
answer = ask_model(test_question)

# 3. SONUÇLARIN EKRANA YAZDIRILMASI
# Soru ve modelin ürettiği cevabı karşılaştırmalı olarak görüntülüyoruz.
print(f"Soru: {test_question}")
print(f"Cevap: {answer}")

Soru: Sunrise Hotel'e giriş ve çıkış saatleri nedir?
Cevap: Sunrise Hotel'in giriş saatleri 14:00, çıkış saatleri 12:00'dir.


**Hücre 8: Hibrit Yanıt Fonksiyonu (RAG + İnce Ayarlı Model Birleşimi)**

**Açıklama:** Bu hücre, projenin temel hedefi olan **Hibrit Yaklaşımı** (Hybrid Approach) hayata geçirir. Bu aşamada, **İnce Ayar** (Fine-tuning) ile kazandırılan "otel personeli üslubu" ile **RAG** sisteminin sunduğu "doğru ve güncel veri" birleştirilir. Süreç şu şekilde işler: Sistem önce kullanıcı sorusuyla ilgili en alakalı metin parçalarını dokümandan çeker; ardından bu verileri modelin önüne bir "referans kaynağı" olarak koyar. Böylece model, hem eğitilmiş dil yeteneklerini kullanır hem de cevabı üretirken kendisine sunulan doküman verilerinden destek alarak hata (halüsinasyon) yapma riskini en aza indirir.

In [9]:
# GELİŞMİŞ HİBRİT YANIT FONKSİYONU (SON RÖTUŞ: TEMİZLİK GARANTİLİ)
def hybrid_ask_model(question):
    """
    Bu fonksiyon hem detaylı cevap verir hem de negatif durumlarda
    gereksiz eklemeleri (leakage) yazılımsal olarak keser.
    """

    # 1. ADIM: Bağlamı getir
    retrieved_context = get_relevant_context(question)

    # Güvenlik: Bağlam boşsa reddet
    if len(retrieved_context.strip()) < 10:
         return "Bu bilgi otel dokümanında bulunmamaktadır."

    # 2. ADIM: Taktiksel Prompt (Mevcut başarılı prompt korunuyor)
    tactical_prompt = """Sen Sunrise Hotel'in kurumsal asistanısın. Aşağıdaki 'Giriş' metnini kaynak olarak kullan ve soruyu cevapla.

KURALLAR:
1. DETAY VER: Cevap metinde varsa, özet geçme! Metindeki tüm sayıları, saatleri (örn: 48 saat, 14:00) ve şartları (örn: No-show) eksiksiz yaz.
2. HALÜSİNASYON GÖRME: Eğer sorulan konu 'Giriş' metninde AÇIKÇA yazmıyorsa, sakın dışarıdan bilgi ekleme.
3. REDDET: Metinde olmayan sorulara sadece "Bu bilgi otel dokümanında bulunmamaktadır." cevabını ver.

### Talimat:
{}

### Giriş (Bağlam):
{}

### Yanıt:
"""

    prompt = tactical_prompt.format(question, retrieved_context)

    inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

    # 3. ADIM: Yanıt üretimi
    outputs = model.generate(
        **inputs,
        max_new_tokens = 300,
        use_cache = True,
        temperature = 0.05,
        repetition_penalty = 1.1
    )

    decoded_output = tokenizer.batch_decode(outputs)[0]
    response = decoded_output.split("### Yanıt:\n")[-1].replace(EOS_TOKEN, "").strip()

    # --- SON RÖTUŞ (CLEANING TRICK) ---
    # Eğer model "Bilgi yok" dediyse, arkasından gelen gevezeliği kesiyoruz.
    refusal_phrase = "Bu bilgi otel dokümanında bulunmamaktadır."

    # Eğer cevap reddetme cümlesini içeriyorsa, sadece o cümleyi döndür ve gerisini sil.
    if refusal_phrase in response:
        return refusal_phrase

    # Temizlik (Varsa #### notlarını sil)
    if "####" in response:
        response = response.split("####")[0].strip()

    return response

print("✅ Hibrit Fonksiyon Hazır: Cevaplar artık %100 temiz ve detaylı.")

✅ Hibrit Fonksiyon Hazır: Cevaplar artık %100 temiz ve detaylı.


**Hücre 9: Performans Değerlendirmesi ve Model Testleri (Evaluation)**

**Açıklama:** Bu hücre, projenin yol haritasında belirtilen 3 ana soru tipine (Basit, Karmaşık ve Negatif) göre modelin performansını test etmek için tasarlanmıştır. Hibrit yaklaşımımızın (RAG + Fine-Tuning) doğruluğu, hazırlanan doküman kapsamındaki 9 farklı senaryo üzerinden sorgulanır. Bu aşama, sistemin bilgi çıkarma, verileri sentezleme ve doküman dışı bilgileri reddetme yeteneklerini doğrular.

In [10]:
# PERFORMANS DEĞERLENDİRME TESTLERİ
# Proje isterlerine uygun olarak 3 farklı kategoride model testi gerçekleştirilmektedir.

test_categories = {
    "Basit Testler (Doğrudan Cevaplar)": [
        "Sunrise Hotel'e giriş ve çıkış saatleri nedir?",
        "Kahvaltı hangi saatler arasında servis ediliyor?",
        "Otelde toplam kaç oda bulunmaktadır?"
    ],
    "Karmaşık Testler (Bilgi Birleştirme)": [
        "Rezervasyon iptalinde iade şartları nelerdir?",
        "Evcil hayvan politikası ve günlük ek ücreti nedir?",
        "Suit odaların standart odalardan farkları nelerdir?"
    ],
    "Negatif Testler (Doküman Dışı Bilgiler)": [
        "Otelinize yakın araç kiralama şirketlerinin fiyatları nedir?",
        "İstanbul Havalimanı'ndan otele uçak bileti fiyatı ne kadar?",
        "Otelin 2026 yılı oda fiyat listesi açıklandı mı?"
    ]
}

print("--- MODEL PERFORMANS DEĞERLENDİRMESİ BAŞLIYOR ---\n")

for category, questions in test_categories.items():
    print(f"=== KATEGORİ: {category} ===")
    for q in questions:
        # Hibrit model fonksiyonumuzu kullanarak yanıt alıyoruz
        answer = hybrid_ask_model(q)
        print(f"Soru  : {q}")
        print(f"Cevap : {answer}")
        print("-" * 30)
    print("\n")

print("--- TEST SÜRECİ TAMAMLANDI ---")

--- MODEL PERFORMANS DEĞERLENDİRMESİ BAŞLIYOR ---

=== KATEGORİ: Basit Testler (Doğrudan Cevaplar) ===
Soru  : Sunrise Hotel'e giriş ve çıkış saatleri nedir?
Cevap : Otele giriş saati 14:00, çıkış saati 11:00'dir.
------------------------------
Soru  : Kahvaltı hangi saatler arasında servis ediliyor?
Cevap : 07:00 – 10:00 saatleri arasında kahvaltı servis ediliyor.
------------------------------
Soru  : Otelde toplam kaç oda bulunmaktadır?
Cevap : 120 oda bulunmaktadır.
------------------------------


=== KATEGORİ: Karmaşık Testler (Bilgi Birleştirme) ===
Soru  : Rezervasyon iptalinde iade şartları nelerdir?
Cevap : Rezervasyon iptalinde iade şartları: Rezervasyon, giriş tarihinden en az 48 saat önce iptal edilirse ön ödeme iade edilir.
------------------------------
Soru  : Evcil hayvan politikası ve günlük ek ücreti nedir?
Cevap : Evcil hayvan politikası: Küçük ırk evcil hayvanlara izin verilmektedir. Günlük ek 20€ temizlik ücreti uygulanır.
------------------------------
Soru  : Su

**Hücre 10: Gradio Web Arayüzü ve Kullanıcı Deneyimi**

**Açıklama:** Bu hücre, projemizin son aşaması olan etkileşimli kullanıcı arayüzünü oluşturur. **Gradio** kütüphanesi kullanılarak tasarlanan bu sohbet arayüzü, kullanıcının otel hakkında merak ettiği soruları yazabileceği ve saniyeler içinde yanıt alabileceği modern bir dijital asistan deneyimi sunar. Sistem, arka planda oluşturduğumuz **Hibrit Yaklaşımı (hybrid_ask_model)** kullanarak hem doküman verilerini tarar hem de eğitilmiş model üslubuyla profesyonel yanıtlar üretir. Ayrıca, share=True parametresi sayesinde, modelin çalışmasını yerel bilgisayar dışında da (örneğin mobil cihazlarda veya başka bilgisayarlarda) test edebilmek için 72 saat geçerli olan genel bir web bağlantısı (URL) oluşturulur.

In [11]:
# 1. GRADIO KÜTÜPHANESİNİN KURULUMU
# Kullanıcı dostu bir web arayüzü oluşturmak için Gradio kütüphanesini yüklüyoruz.
!pip install gradio

import gradio as gr

# 2. ARAYÜZ MANTIĞI (MESSAGE HANDLER)
# Kullanıcı bir mesaj gönderdiğinde bu fonksiyon tetiklenir.
def chat_response(message, history):
    # Doküman araması (RAG) ve eğitilmiş ağırlıkları (Fine-Tuning) birleştiren
    # hibrit fonksiyonumuzu çağırarak nihai yanıtı kullanıcıya döndürüyoruz.
    return hybrid_ask_model(message)

# 3. KULLANICI ARAYÜZÜ TASARIMI
# Sohbet penceresinin başlığı, açıklaması, örnek soruları ve görsel teması belirlenir.
interface = gr.ChatInterface(
    fn=chat_response,
    title="🏨 Sunrise Hotel Akıllı Asistanı",
    # Projenin teknik içeriğini özetleyen açıklama metni.
    description="RAG + Fine-Tuned Llama-3 Modeli ile Hibrit Otel Destek Sistemi",
    # Kullanıcıya rehberlik etmesi amacıyla sunulan hazır örnek sorular.
    examples=[
        ["Sunrise Hotel'e giriş ve çıkış saatleri nedir?"],
        ["Rezervasyonumu iptal edersem paramı geri alabilir miyim?"],
        ["Kahvaltı saatleri nedir?"],
        ["Otelde evcil hayvan kabul ediliyor mu?"],
        ["Spa ve havuz kullanım saatleri nedir?"],
        ["Hangi ödeme yöntemlerini kabul ediyorsunuz?"]
    ],
    theme="soft" # Kullanıcı deneyimini iyileştiren yumuşak bir tema seçilmiştir.
)

# 4. ARAYÜZÜN BAŞLATILMASI VE PAYLAŞILMASI
# 'share=True' parametresi, projeyi sunum sırasında başkalarıyla paylaşabilmek için genel bir link sağlar.
# 'debug=True' parametresi ise olası hataları doğrudan Notebook üzerinden izlememize olanak tanır.
interface.launch(share=True, debug=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://beeae7eaeb68f1e80f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://beeae7eaeb68f1e80f.gradio.live
